# What you read *last* matters

You just finished **volume 2** of a series. Should the recommender care that you read it *last*?

Every path we have built so far says no. Item-item CF (Unit 4), matrix factorization, the two-tower,
the reranker -- each one treats your history as a **bag**: the scoreboard hands a path the set
`context["seen"]`, and a set has no order. Volume 2 read yesterday and volume 2 read three years ago
look exactly the same. So does a reader who finished a series and a reader who is halfway through it.

But real reading is a **sequence**. You read volume 3 *after* volume 2. You follow an author for a few
books and then drift. A rainy week puts you in a mystery mood. This unit asks: **if the recommender
looks at the order of your history -- and especially at what you read most recently -- does it
recommend better?** We answer it three ways, from the cheapest to the most flexible:

1. **Recency** -- query item-item CF with only your *last few* books instead of all of them.
2. **Item transitions** -- count what readers read *right after* each book (a Markov chain).
3. **Self-attention** -- a tiny SASRec-style neural model that learns which past reads to look at.

And we will prove, with a control experiment, that it is the *order* that helps -- not just the books.

## 1. New data: the session log

Unit 12 runs on a **different log**. The main interaction log turned out to carry almost no order
signal (a capped SASRec on it scored *below* plain item-item CF and the same as a shuffled-history
copy), so the generator ships a separate, seeded **session log**, `sessions.csv.gz`. It has the same
columns as `interactions.csv.gz` (plus `session_id`), the same catalog, and the same taste-aware
exposure -- but its own synthetic **readers**, with longer histories and real order effects: a forced
"next volume" exposure after a series read, an author-follow streak that decays, and a genre mood
that persists through a session.

**Read every number in this unit as "on the session log".** It is a different reader population, so a
hit@10 here is **not comparable** to the main-log numbers of Units 2-11 (item-item CF scored about 0.25
there; here it will score much higher, because these histories are longer and more predictable -- not
because CF got better).

A second file, `series.csv.gz`, says which books are volumes of the same series (`item_id, series_id,
volume`). We use it **only as a diagnostic lens** -- to *read* what a path recommends -- never as a
model input. No path in this unit is told which books form a series; it has to discover that from the
order of reads. As always, `test` stays sealed: we score on `val` only.

In [1]:
import time
from collections import defaultdict

import numpy as np
import pandas as pd
from bookrec import (
    Candidate,
    ItemItemRetrievalPath,
    LastKItemItemPath,
    PopularityRetrievalPath,
    SasrecRetrievalPath,
    TransitionRetrievalPath,
    catalog_coverage,
    first_unseen_val,
    generated_dir,
    item_item_cosine,
    load_catalog,
    load_series,
    load_train_sequences,
    rank,
    run_validation_scoreboard,
)

K = 10
SHUFFLE_SEED = 20260930 + 1000   # the order-shuffled control's seed (committed data seed + 1000)

data_dir = generated_dir()
catalog = load_catalog(data_dir / "catalog.csv.gz")
catalog_ids = sorted(catalog)
sessions_path = data_dir / "sessions.csv.gz"
rows = pd.read_csv(sessions_path).to_dict("records")       # reader_id, item_id, session_id, timestamp, split, label
series = load_series(data_dir / "series.csv.gz")            # {item_id: (series_id, volume)} -- diagnostics only
histories = load_train_sequences(sessions_path)             # {reader_id: [train positives, oldest -> newest]}

lengths = [len(seq) for seq in histories.values()]
print(f"session log: {len(rows)} rows, {len(histories)} readers with a train history")
print(f"train history length: median {int(np.median(lengths))}, max {max(lengths)} positives")
print(f"series table: {len(series)} books in {len({s for s, _ in series.values()})} series (diagnostics only)")

session log: 181802 rows, 1500 readers with a train history
train history length: median 28, max 61 positives
series table: 1002 books in 263 series (diagnostics only)


`load_train_sequences` is the one place the order comes from: each reader's **train positives only**,
sorted by `(timestamp, row order)`. Val and test rows never enter a history. Here is one reader's
history, newest last, with the series table used to *label* the books (the model never sees these
labels). Reader 35's most recent read is **volume 2 of series 119** -- the hook's situation exactly.

In [2]:
reader = 35
history = histories[reader]


def label(item_id):
    # Diagnostic label only: "series:volume" from series.csv.gz, or "-" for a stand-alone book.
    if item_id in series:
        series_id, volume = series[item_id]
        return f"series {series_id} vol {volume}"
    return "-"


print(f"reader {reader}: {len(history)} train reads; the last 8, oldest -> newest:")
for position in range(len(history) - 8, len(history)):
    item_id = history[position]
    print(f"  #{position:>2}  book {item_id:>4}  {label(item_id)}")

reader 35: 24 train reads; the last 8, oldest -> newest:
  #16  book 1540  -
  #17  book  916  -
  #18  book 1800  series 122 vol 2
  #19  book 1296  series 122 vol 3
  #20  book 1972  -
  #21  book  799  series 51 vol 2
  #22  book  938  -
  #23  book 1364  series 119 vol 2


Look at the order. Volumes 2 and 3 of series 122 arrive back to back (volume 1 came a few reads
earlier); then a few other books; then volume 2 of series 119. The same 24 books in any other order
would look identical to Unit 4's CF. The question for every path below: **does reader 35's top-10 contain volume 3 of
series 119** (book 1147, which reader 35 really did read next, in `val`)?

## 2. The bag baseline, then recency

First the reference points, scored with the unchanged Unit-1 scoreboard -- just pointed at the session
log. Popularity is the floor; Unit 4's `ItemItemRetrievalPath` is the bag model we have to beat.

In [3]:
popularity = PopularityRetrievalPath().fit(rows, catalog=catalog_ids)
item_item = ItemItemRetrievalPath().fit(rows, catalog=catalog_ids)

pop_board = run_validation_scoreboard(popularity, sessions_path, catalog_ids=catalog_ids, k=K)
bag_board = run_validation_scoreboard(item_item, sessions_path, catalog_ids=catalog_ids, k=K)
print(f"on the session log ({bag_board.readers} val readers), hit@10:")
print(f"  popularity          {pop_board.hit_rate_at_k:.4f}")
print(f"  item-item CF (bag)  {bag_board.hit_rate_at_k:.4f}")

on the session log (1419 val readers), hit@10:
  popularity          0.1135
  item-item CF (bag)  0.4799


Item-item CF scores about **0.480**, over 4x popularity (0.114). Every later path in this unit is
compared against that bag number.

To compare paths **reader by reader** -- and to read next-item hits, coverage and the series diagnostic
off the *same* recommendations -- we keep each path's top-10 per reader. `top10` applies exactly the
scoreboard's rule: the cohort is every reader with a `val` positive they have not already read in
train; the seen set (train positives) is excluded; `rank` keeps the top 10. A per-reader hit is 1 when
any unseen `val` book is in that list. Its mean must reproduce the scoreboard -- we check it.

In [4]:
seen_by = defaultdict(set)
val_by = defaultdict(set)
for row in rows:
    if row["label"] == 1 and row["split"] == "train":
        seen_by[row["reader_id"]].add(row["item_id"])
    elif row["label"] == 1 and row["split"] == "val":
        val_by[row["reader_id"]].add(row["item_id"])
cohort = [r for r in sorted(val_by) if val_by[r] - seen_by[r]]


def top10(path):
    # Each cohort reader's top-10, by the scoreboard's rule (seen excluded, rank keeps 10).
    return {
        r: [c.item_id for c in rank(path.retrieve(r, {"seen": seen_by[r]}, K), n=K, exclude=seen_by[r])]
        for r in cohort
    }


def hits_of(tops):
    # Per-reader hit@10 (1.0 if any unseen val book is in the top-10), in cohort order.
    return np.array([float(bool(set(tops[r]) & (val_by[r] - seen_by[r]))) for r in cohort])


def paired(a, b):
    # Mean per-reader difference a - b and its paired standard error.
    diff = a - b
    return diff.mean(), diff.std(ddof=1) / np.sqrt(len(diff))


tops = {"item-item": top10(item_item)}
hits = {"item-item": hits_of(tops["item-item"])}
assert abs(hits["item-item"].mean() - bag_board.hit_rate_at_k) < 1e-12
print(f"{len(cohort)} cohort readers; per-reader mean {hits['item-item'].mean():.4f} == scoreboard")
print(f"reader {reader}: book 1147 (series 119 vol 3) in the bag top-10? {1147 in tops['item-item'][reader]}")

1419 cohort readers; per-reader mean 0.4799 == scoreboard
reader 35: book 1147 (series 119 vol 3) in the bag top-10? False


### Recency by hand: query with your last 3 books

The bag query scores a candidate by its summed similarity to **every** book you have read:
$\mathrm{score}(c) = \sum_{s \in \mathrm{seen}} \mathrm{sim}(c, s)$. With 24 books in the bag, volume 2
of series 119 is one vote in 24. **Recency** keeps the same similarity matrix but sums over only your
**last $k$ distinct** reads -- the most recent occurrence of each book (re-reads are common on this log,
so a re-read counts once, at its newest position):

$$\mathrm{score}(c) = \sum_{s \,\in\, \mathrm{last}_k} \mathrm{sim}(c, s)$$

We build Unit 4's cosine similarity by hand from the train positives and query it with reader 35's last
3 distinct books.

In [5]:
col_of = {item_id: col for col, item_id in enumerate(catalog_ids)}
incidence = np.zeros((len(histories), len(catalog_ids)))
for row_index, r in enumerate(sorted(histories)):
    for item_id in histories[r]:
        incidence[row_index, col_of[item_id]] = 1.0
sim = item_item_cosine(incidence)                       # Unit 4's item x item cosine

last3 = []                                              # last 3 DISTINCT reads, newest first
for item_id in reversed(history):
    if item_id not in last3:
        last3.append(item_id)
    if len(last3) == 3:
        break

scores = sim[:, sorted(col_of[i] for i in last3)].sum(axis=1)
by_hand = rank(
    [Candidate(item_id, float(scores[col]), "last-3 by hand") for col, item_id in enumerate(catalog_ids)
     if scores[col] > 0],
    n=K, exclude=seen_by[reader],
)
print(f"reader {reader}'s last 3 distinct reads (newest first): {[(i, label(i)) for i in last3]}")
print(f"last-3 top-10 by hand: {[c.item_id for c in by_hand]}")
print(f"book 1147 (series 119 vol 3) in it? {1147 in [c.item_id for c in by_hand]}")

reader 35's last 3 distinct reads (newest first): [(1364, 'series 119 vol 2'), (938, '-'), (799, 'series 51 vol 2')]
last-3 top-10 by hand: [331, 1147, 1562, 465, 963, 1172, 615, 1857, 1974, 47]
book 1147 (series 119 vol 3) in it? True


Volume 3 is there. Reader 35's last three reads point *at* the series, and nothing older drowns them out.

### Reveal `LastKItemItemPath`

`bookrec.LastKItemItemPath(k_last=3)` is exactly this: Unit 4's cosine, queried with the reader's last
`k_last` distinct train positives. Its `fit` stores each reader's **ordered** history (the same
`load_train_sequences` order), and `retrieve` queries with that stored history -- `context["seen"]` is
only the exclusion set, because a set has no "last". (`k_last=None` means "all distinct books" and
reproduces the bag path exactly.) It matches our by-hand list, and then the scoreboard:

In [6]:
last_k = LastKItemItemPath(k_last=3).fit(rows, catalog=catalog_ids)
assert [c.item_id for c in last_k.retrieve(reader, {"seen": seen_by[reader]}, K)] == [c.item_id for c in by_hand]
print("shipped LastKItemItemPath(3) == by-hand last-3 query for reader", reader)

tops["last-3 CF"] = top10(last_k)
hits["last-3 CF"] = hits_of(tops["last-3 CF"])
lift, se = paired(hits["last-3 CF"], hits["item-item"])
print(f"hit@10  item-item (bag) {hits['item-item'].mean():.4f}   last-3 CF {hits['last-3 CF'].mean():.4f}")
print(f"recency lift {lift:+.4f}  (paired SE {se:.4f}, {lift / se:.1f} SE)")

shipped LastKItemItemPath(3) == by-hand last-3 query for reader 35


hit@10  item-item (bag) 0.4799   last-3 CF 0.5511
recency lift +0.0712  (paired SE 0.0136, 5.3 SE)


Forgetting most of the history **helps**: last-3 CF scores about **0.551**, a **+0.071** lift over the
bag (about 5 paired standard errors -- not noise). The same similarity matrix, the same books; only the
*query* changed. (Exercise 1 sweeps `k_last` -- is 3 the best window?)

## 3. Item transitions: what do readers read *right after* this book?

Recency still asks "what is *similar* to your recent books?" -- and similarity is symmetric: volume 2 is
as similar to volume 3 as volume 3 is to volume 2. A **transition** model asks a directed question:
across all readers, **which book comes next after this one?** Count it. For every reader's ordered
train history $x_1, x_2, \dots, x_n$, every consecutive pair $(x_t, x_{t+1})$ adds one to a count
table:

$$T[a, b] = \#\{\text{times } b \text{ was read right after } a\}$$

This is a **first-order Markov chain**: the next read depends only on the current one. To recommend, sum
the $T$ rows of the reader's last 5 reads and take the top 10.

In [7]:
T = np.zeros((len(catalog_ids), len(catalog_ids)))
for seq in histories.values():
    for a, b in zip(seq[:-1], seq[1:]):                  # consecutive pairs, in order
        T[col_of[a], col_of[b]] += 1.0

print(f"{int(T.sum())} consecutive train pairs counted")
after = T[col_of[1364]]                                 # what comes right after series 119 vol 2?
print("most common next reads after book 1364 (series 119 vol 2):")
for col in np.argsort(-after, kind="stable")[:4]:
    print(f"  book {catalog_ids[col]:>4}  {int(after[col])} time(s)  {label(catalog_ids[col])}")

40381 consecutive train pairs counted
most common next reads after book 1364 (series 119 vol 2):
  book 1147  12 time(s)  series 119 vol 3
  book 1974  6 time(s)  series 3 vol 4
  book 1637  5 time(s)  series 56 vol 4
  book  963  4 time(s)  series 4 vol 3


The chain discovered the series on its own: the most common read after volume 2 is **volume 3**. Nobody
told it book 1147 is volume 3 -- readers' *order* did.

### Reveal `TransitionRetrievalPath`

`bookrec.TransitionRetrievalPath(k_last=5)` counts the same table in `fit` (from train positives only)
and scores a reader by the summed `T` rows of their last 5 stored reads. Its table equals ours.

In [8]:
transition = TransitionRetrievalPath(k_last=5).fit(rows, catalog=catalog_ids)
assert np.allclose(transition.transitions, T)
print("shipped transition table == by-hand counts")

tops["transition"] = top10(transition)
hits["transition"] = hits_of(tops["transition"])
print(f"hit@10  transition {hits['transition'].mean():.4f}   (item-item bag {hits['item-item'].mean():.4f})")
print(f"reader {reader}: book 1147 in the transition top-10? {1147 in tops['transition'][reader]}")

shipped transition table == by-hand counts


hit@10  transition 0.5159   (item-item bag 0.4799)
reader 35: book 1147 in the transition top-10? True


### The control: shuffle the order, keep the books

Transition scores about **0.516** -- above the bag. But is that really *order*? A transition model
also learns which books **co-occur** in the same histories, which a bag model knows too. To separate the
two we run a **control experiment**: shuffle every reader's history (a seeded random permutation),
keeping exactly the same books, and refit. The shuffle must apply to **both** what the model learns from
*and* the history it queries with -- otherwise we are not testing order, we are testing a mismatch.
`shuffle_seed` does exactly that. If order carries the lift, the shuffled copy should collapse; if the
lift were only "which books go together", it should survive.

In [9]:
shuffled = TransitionRetrievalPath(k_last=5, shuffle_seed=SHUFFLE_SEED).fit(rows, catalog=catalog_ids)
assert sorted(shuffled.histories[reader]) == sorted(history)    # same books ...
assert shuffled.histories[reader] != history                    # ... different order

tops["transition (shuffled)"] = top10(shuffled)
hits["transition (shuffled)"] = hits_of(tops["transition (shuffled)"])
drop, se = paired(hits["transition"], hits["transition (shuffled)"])
ordered_hit, shuffled_hit = hits["transition"].mean(), hits["transition (shuffled)"].mean()
print(f"transition ordered {ordered_hit:.4f}   order-shuffled {shuffled_hit:.4f}")
print(f"ordered - shuffled {drop:+.4f} (paired SE {se:.4f}, {drop / se:.0f} SE); ratio {ordered_hit / shuffled_hit:.2f}x")
print(f"shuffled control even falls below the bag ({hits['item-item'].mean():.4f})")

transition ordered 0.5159   order-shuffled 0.3326
ordered - shuffled +0.1832 (paired SE 0.0140, 13 SE); ratio 1.55x
shuffled control even falls below the bag (0.4799)


The lift **collapses**: from about 0.516 to about **0.333**, a drop of **+0.18** (1.55x) -- well below
even the bag model. Same books, same counting code; the only thing destroyed was the order. That is the
evidence that **order** -- not the item set -- carries the transition path's lift. A raw hit@10 can never
tell you *why* a model works; a control that removes exactly one ingredient can.

## 4. Self-attention: let the model choose which past reads to look at

Last-k and transitions hard-code *how* to use the order: "sum the last 3" or "count the next one". A
**self-attention** model *learns* it. SASRec (self-attentive sequential recommendation) reads the history
as a sequence of item embeddings plus **position** embeddings, and at each position it builds a weighted
mix of the earlier positions:

- each position $t$ emits a **query** $q_t$, a **key** $k_t$ and a **value** $v_t$ (three learned linear
  maps of its embedding);
- position $t$ scores every position $j$ by $q_t \cdot k_j / \sqrt{d}$, turns those scores into weights
  with a **softmax**, and outputs the weighted sum of the values $v_j$;
- a **causal mask** forbids looking ahead: position $t$ may attend only to positions $j \le t$. At
  training time the model predicts the item at $t + 1$ from positions $1..t$, so peeking at $t + 1$ would
  be cheating.

The final position's output is the reader's state; every book is scored by its **dot product** with that
state (the same embedding table, "tied"), exactly the Unit-7/8 embedding-retrieval head. Here is one
attention step by hand, on a toy history of 4 books with 4-dimensional embeddings.

In [10]:
rng = np.random.default_rng(0)
d = 4
toy = ["series vol 1", "cookbook", "series vol 2", "mystery"]
items = rng.normal(size=(len(toy), d))            # one embedding per book
positions = 0.1 * rng.normal(size=(len(toy), d))  # one embedding per position (0..3)
x = items + positions
w_q, w_k, w_v = (rng.normal(size=(d, d)) for _ in range(3))


def attend(x):
    q, k, v = x @ w_q, x @ w_k, x @ w_v
    scores = q @ k.T / np.sqrt(d)
    future = np.triu(np.ones((len(x), len(x)), dtype=bool), k=1)   # True above the diagonal = the future
    scores = np.where(future, -np.inf, scores)                       # causal mask
    weights = np.exp(scores - scores.max(axis=1, keepdims=True))
    weights = weights / weights.sum(axis=1, keepdims=True)          # row-wise softmax
    return weights @ v, weights


out, weights = attend(x)
print("causal mask (1 = may attend):")
print((~np.triu(np.ones((4, 4), dtype=bool), k=1)).astype(int))
print("attention weights (row t = how position t mixes positions 0..t):")
print(np.round(weights, 3))

causal mask (1 = may attend):
[[1 0 0 0]
 [1 1 0 0]
 [1 1 1 0]
 [1 1 1 1]]
attention weights (row t = how position t mixes positions 0..t):
[[1.    0.    0.    0.   ]
 [0.453 0.547 0.    0.   ]
 [0.615 0.379 0.006 0.   ]
 [0.407 0.59  0.003 0.   ]]


Each row sums to 1 and is zero above the diagonal: position 0 can only look at itself, the last position
spreads its attention over the whole history. **Causality** is a property we can test: change the last
book, and no earlier position's output may move.

In [11]:
changed = x.copy()
changed[-1] = rng.normal(size=d)                 # a different last read
out_changed, _ = attend(changed)
assert np.allclose(out_changed[:-1], out[:-1])   # positions 0..2 cannot see position 3
assert not np.allclose(out_changed[-1], out[-1])
print("changing the last book moved only the last position's output: the mask works")

item_table = rng.normal(size=(6, d))             # a toy 6-book embedding table
print("next-book scores = item_table @ final state:", np.round(item_table @ out[-1], 2))

changing the last book moved only the last position's output: the mask works
next-book scores = item_table @ final state: [-0.54 -1.94  0.15 -1.21 -0.08 -0.08]


### Reveal `SasrecRetrievalPath`

`bookrec.SasrecRetrievalPath` wraps this step in a real (tiny) model, trained in PyTorch: learned item
and position embeddings over the last 50 reads, **one** pre-LayerNorm causal self-attention block with
one head (`torch.nn.MultiheadAttention` with a `triu` mask, exactly our `future` matrix) and a small
feed-forward layer, both with residual connections; a tied output embedding; and a full-softmax
cross-entropy loss that asks **every position** to predict the next book. The size is capped by design
(dimension 32, one block, 50 reads). Its defaults are the unit's frozen fast recipe: 30 epochs, batch
128, Adam at learning rate 2e-3, dropout 0.2, seed 0, single-threaded and deterministic. After training,
`retrieve` runs a **numpy** forward pass (`final_state`) -- torch is only needed to train.

This is the one expensive cell in the unit: a single fit, well under a minute and a half.

In [12]:
start = time.perf_counter()
sasrec = SasrecRetrievalPath().fit(rows, catalog=catalog_ids)
seconds = time.perf_counter() - start
loss = sasrec.loss_history
print(f"SASRec (recipe D) trained in {seconds:.0f} s: {len(loss)} epochs, loss {loss[0]:.2f} -> {loss[-1]:.2f}")
print(f"reader {reader}'s final hidden state: {sasrec.final_state(history).shape} (numpy forward)")

SASRec (recipe D) trained in 46 s: 30 epochs, loss 7.30 -> 5.06
reader 35's final hidden state: (32,) (numpy forward)


In [13]:
tops["SASRec"] = top10(sasrec)
hits["SASRec"] = hits_of(tops["SASRec"])
gap, se = paired(hits["SASRec"], hits["item-item"])
print(f"hit@10  SASRec {hits['SASRec'].mean():.4f}   item-item (bag) {hits['item-item'].mean():.4f}")
print(f"SASRec - item-item {gap:+.4f}  (paired SE {se:.4f}, {gap / se:.1f} SE)")

# Order evidence for SASRec: a second (order-shuffled) SASRec fit would double this cell's cost, so the
# lesson QUOTES it. Source: the Unit-12 milestone (projects/bookrec/milestones/unit-12-sequence-aware.ipynb),
# gate (c) cell d83a2ac8 -- `sasrec_order`, `sasrec_order_se` -- measured at this exact recipe and seed
# and asserted there in CI.
SASREC_ORDERED_MINUS_SHUFFLED, SASREC_ORDER_SE = 0.040, 0.0105
print(f"cited: SASRec ordered - shuffled {SASREC_ORDERED_MINUS_SHUFFLED:+.3f} "
      f"({SASREC_ORDERED_MINUS_SHUFFLED / SASREC_ORDER_SE:.1f} SE), measured + asserted by the U12 milestone")

hit@10  SASRec 0.4228   item-item (bag) 0.4799
SASRec - item-item -0.0571  (paired SE 0.0121, -4.7 SE)
cited: SASRec ordered - shuffled +0.040 (3.8 SE), measured + asserted by the U12 milestone


Two facts, and both matter:

- **SASRec learns order.** Its order-shuffled copy (same recipe, histories permuted for training and
  query) scores about 0.383; the ordered model about 0.423: **+0.040, 3.8 paired SE**. We cite that
  number rather than re-measure it here -- a second fit would double the lesson's cost -- and it is not
  a borrowed claim: this unit's milestone notebook trains both models at this exact recipe and asserts
  the gap in CI. The live control *you* ran is the transition one in Section 3.
- **SASRec trails the bag.** At this recipe it scores about **0.423**, about **0.06 below** plain
  item-item CF (-0.057, roughly 5 paired SE). That is not a tie.

## 5. The honest scoreboard

One table, read off the same top-10 lists. Besides hit@10 (*any* unseen `val` book in the top 10) we
record **next-item hit@10** -- is the reader's *first* new `val` book in the top 10? (`first_unseen_val`)
-- and **catalog coverage**, the fraction of the catalog that appears in at least one reader's top 10.

In [14]:
first_next = first_unseen_val(sessions_path)    # {reader: their first val book not read in train}
print(f"{'path (session log, val)':<24} {'hit@10':>7} {'next-item':>10} {'coverage':>9}")
print(f"{'popularity':<24} {pop_board.hit_rate_at_k:>7.3f} {'':>10} {'':>9}")
for name in tops:
    next_hit = np.mean([float(first_next[r] in tops[name][r]) for r in cohort if r in first_next])
    coverage = catalog_coverage(tops[name].values(), catalog_ids)
    print(f"{name:<24} {hits[name].mean():>7.3f} {next_hit:>10.3f} {coverage:>9.3f}")

path (session log, val)   hit@10  next-item  coverage
popularity                 0.113                     
item-item                  0.480      0.245     0.336
last-3 CF                  0.551      0.369     0.713
transition                 0.516      0.298     0.369
transition (shuffled)      0.333      0.125     0.351
SASRec                     0.423      0.176     0.284


### A diagnostic lens: does the path recommend the next volume?

Now -- and only now, *after* every model is trained -- we use `series.csv.gz` to ask a sharp question.
Collect every **next-in-series event**: a reader has volume $v$ of a series among their last 5 train
reads, and volume $v + 1$ is an unseen `val` book. For each path, how often is volume $v + 1$ in the
reader's top 10? The table never touched any model; it only labels the answers.

In [15]:
volume_of = {(s, v): item_id for item_id, (s, v) in series.items()}
events = []                                      # (reader, volume-v book, volume-v+1 book)
for r in cohort:
    for item_id in sorted(set(histories[r][-5:])):
        if item_id in series:
            s, v = series[item_id]
            nxt = volume_of.get((s, v + 1))
            if nxt is not None and nxt in val_by[r] - seen_by[r]:
                events.append((r, item_id, nxt))

print(f"{len(events)} next-in-series events (diagnostic only):")
for name in tops:
    rate = np.mean([float(nxt in tops[name][r]) for r, _, nxt in events])
    print(f"  {name:<24} next volume in top-10: {rate:.3f}")

654 next-in-series events (diagnostic only):
  item-item                next volume in top-10: 0.301
  last-3 CF                next volume in top-10: 0.694
  transition               next volume in top-10: 0.534
  transition (shuffled)    next volume in top-10: 0.064
  SASRec                   next volume in top-10: 0.128


## 6. Reading it honestly

**The winners on this data are the cheap order-aware paths.** Last-3 CF (about 0.551) and transition
(about 0.516) both beat the bag (0.480). Last-3 CF is also the best next-item predictor (about 0.37) and
the best next-volume finder (about 0.69 of series events, against 0.30 for the bag) -- and, as a bonus,
it spreads its picks over far more of the catalog (coverage about 0.71). The transition path's lift is
*proved* to be order by its shuffled control (0.516 -> 0.333).

**SASRec does not beat the simple paths here.** At the unit's fast recipe it **trails item-item CF by
about 0.06** (-0.057 on the committed seed) and finds the next volume only about 13% of the time. Be
precise about what that does and does not say:

- It *learns* order (+0.040 over its shuffled copy, 3.8 SE) -- the order signal is reaching it, just not
  enough of it.
- The earlier session-log study (recsys-014) called SASRec a **near-tie** with item-item CF. That claim
  belongs to a different, longer recipe (about 5 minutes per fit) **and** to held-out data seeds, where
  its gap was within 0.04; even that recipe was -0.046 on the committed seed. It is **not** a claim
  about the recipe you just ran, and we do not borrow it.

**Why does counting win?** The order effects in this log are **short-range and item-to-item**: volume
$v$ -> volume $v + 1$, one author's next book, today's genre mood. A count table captures "after book a
comes book b" *directly*, one cell per pair, from every reader at once. A 1-block, 32-dimensional
attention model has to *learn* those same specific pairs through shared embeddings for about 1,700
books from about 1,500 readers' histories in 30 epochs -- its extra flexibility (looking at any earlier
read, with learned weights) has nothing to spend itself on when the useful pattern is "the last book",
and too little data to sharpen 1,700 item embeddings to the precision a count table gets for free.
Attention earns its cost on longer-range patterns, much more data and bigger models -- not on a
next-volume rule.

## Where this goes

- A reader's history is a **sequence**, not a bag. On the session log, querying with only the **last 3
  distinct** books lifts item-item CF from about 0.480 to about 0.551 (+0.071, about 5 SE).
- A first-order **item-transition** table -- "what is read right after this book" -- reaches about 0.516
  and recovers series order unsupervised. Its **order-shuffled control** (shuffle both what it learns
  and what it queries) collapses it to about 0.333: order, not the item set, carries the lift.
- A tiny **causal self-attention** model (SASRec) learns order (+0.040 over its shuffled copy, asserted
  by the milestone) but **trails** the bag by about 0.06 at our size and time budget. Counting wins when
  the order effects are short and item-to-item.
- Every number here is **on the session log** -- a different reader population from Units 2-11 -- and
  `series.csv.gz` was a diagnostic lens, never a model input.

### The bridge ahead

**Unit 13** steps back and evaluates the **whole system** -- beyond accuracy, popularity and exposure
bias, feedback loops, filter bubbles and fairness, and the limits of offline metrics like the hit@10 we
have chased all book -- and ends Part 2's middle stretch with **Checkpoint B**. (The sequence path is
scored on the session log there too.) **Unit 14** is the **capstone**: the full multi-path retrieval +
blend + neural-rerank recommender, benchmarked against the Part-1 build with an ablation table and a
beyond-accuracy table.